In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [2]:
disprot = pd.read_csv("../data/disprot.tsv", sep='\t', low_memory=False)
disprot = disprot.rename(columns={
    'UniProt ACC':'acc', 'Organism':'organism',
    'Term namespace':'term_namespace', 'Term ID':'term',
    'Start':'start', 'End':'end', 'Region ID':'region_id',
})
disprot['acc'] = disprot['acc'].str.split('-').str[0]
human = disprot[disprot['organism'] == 'Homo sapiens'].copy()

# Structural state regions become the ROWS
state = human[human['term_namespace'] == 'Structural state'].copy()
state['region_length'] = state['end'] - state['start'] + 1
state = state[['acc', 'region_id', 'start', 'end', 'region_length']].reset_index(drop=True)

# d2o annotations (Structural transition, IDPO:0000011) become the LABELS
d2o = human[human['term'] == 'IDPO:0000011'].copy()
d2o = d2o[['acc', 'start', 'end']].rename(columns={'start':'d2o_start', 'end':'d2o_end'})

print(f"Structural state regions: {len(state)}")
print(f"d2o annotations: {len(d2o)}")

Structural state regions: 3231
d2o annotations: 299


In [3]:
def compute_overlap(reg_row, d2o_annots):
    """Max fraction of reg_row's residues that fall within any d2o annotation."""
    if len(d2o_annots) == 0:
        return 0.0
    reg_start, reg_end = reg_row['start'], reg_row['end']
    reg_len = reg_end - reg_start + 1
    best = 0.0
    for _, ann in d2o_annots.iterrows():
        overlap_start = max(reg_start, ann['d2o_start'])
        overlap_end   = min(reg_end,   ann['d2o_end'])
        overlap_len   = max(0, overlap_end - overlap_start + 1)
        best = max(best, overlap_len / reg_len)
    return best

overlaps = []
for _, r in state.iterrows():
    d2o_this = d2o[d2o['acc'] == r['acc']]
    overlaps.append(compute_overlap(r, d2o_this))
state['overlap_fraction'] = overlaps
state['d2o_label'] = (state['overlap_fraction'] >= 0.5).astype(int)

print(f"\nRegion-level label distribution:")
print(state['d2o_label'].value_counts())
print(f"Positive rate: {state['d2o_label'].mean():.3f}")

# How many regions per protein
n_per_protein = state.groupby('acc').size().rename('n_regions_in_protein')
state = state.merge(n_per_protein, on='acc')
print(f"\nRegions per protein: median={n_per_protein.median()}, "
      f"mean={n_per_protein.mean():.2f}, max={n_per_protein.max()}")

# Join cluster IDs (inherited from protein)
clusters = pd.read_csv("../data/clusters.csv")
state = state.merge(clusters, on='acc', how='left')
print(f"\nRegions with valid cluster: {state['cluster'].notna().sum()} / {len(state)}")

state.to_csv("../data/regions_master.csv", index=False)
print(f"\nSaved regions_master.csv: {state.shape}")


Region-level label distribution:
d2o_label
0    2853
1     378
Name: count, dtype: int64
Positive rate: 0.117

Regions per protein: median=2.0, mean=2.53, max=38

Regions with valid cluster: 3231 / 3231

Saved regions_master.csv: (3231, 9)


In [4]:
print("\n=== Region-level d2o positive rate breakdown ===")
print(f"Overall: {state['d2o_label'].mean():.3f}")
print(f"Among proteins that have any d2o positive region:")
protein_pos = state.groupby('acc')['d2o_label'].max()
n_proteins_with_pos = protein_pos.sum()
print(f"  Proteins with at least one positive region: {n_proteins_with_pos} "
      f"(protein-level W6/W7 count was 188)")
print(f"Median positive-region overlap fraction (positives only): "
      f"{state[state['d2o_label']==1]['overlap_fraction'].median():.2f}")


=== Region-level d2o positive rate breakdown ===
Overall: 0.117
Among proteins that have any d2o positive region:
  Proteins with at least one positive region: 163 (protein-level W6/W7 count was 188)
Median positive-region overlap fraction (positives only): 1.00
